In [ ]:
import random
import json
import os
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import make_llm_request, adapt_model_kwargs_for_model
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_types import carry_out_unblind_experiment
from local_variables import TIME_SERIES_CATEGORIES, POLITICAL_ATTITUDES_CATEGORIES

In [ ]:
models = ["gpt-5-mini"]
n = 20
custom_model_kwargs = {}
path_to_save_model_outputs = "./unblind_experiment"
random_seed = 41
experiment_name = "evaluate_time_series_trends"

In [ ]:
# Build DataFrame from individual CSV files.
# For each row, a political_pole and political_attitude are randomly assigned.
# The unblinded prompt reveals the think-tank's political_attitude.
random.seed(random_seed)
n_files_per_category = 10
rows = []
for time_series_category in TIME_SERIES_CATEGORIES:
    for trend_direction in ["up", "down"]:
        for i in range(1, n_files_per_category + 1):
            file_path = f"./data/{time_series_category}_{trend_direction}_{i}.csv"
            if os.path.exists(file_path):
                time_series_df = pd.read_csv(file_path)
                time_series_data = time_series_df.to_string(index=False)
                political_attitude_category = random.choice(list(POLITICAL_ATTITUDES_CATEGORIES.keys()))
                political_pole = random.choice(["right", "left"])
                political_attitude = POLITICAL_ATTITUDES_CATEGORIES[political_attitude_category][political_pole]
                rows.append({
                    "time_series_category": time_series_category,
                    "time_series_data": time_series_data,
                    "trend_direction": trend_direction,
                    "file_id": f"{time_series_category}_{trend_direction}_{i}",
                    "political_pole": political_pole,
                    "political_attitude_category": political_attitude_category,
                    "political_attitude": political_attitude,
                })

df = pd.DataFrame(rows)
df

In [ ]:
# The unblinded prompt reveals which politically-aligned think tank claims the trend.
system_prompt = EXPERIMENTS[experiment_name]["unblind_experiment"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["unblind_experiment"]["user_prompt_template"]
variables = ["political_attitude", "time_series_category", "trend_direction", "time_series_data"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
# Test a single request
model_name = models[0]
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
user_prompt = user_prompt_template.format(**{var: df.loc[0, var] for var in variables})
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)

In [ ]:
payloads = await carry_out_unblind_experiment(
    models=models,
    df=df,
    variables=variables,
    n=n,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
    custom_model_kwargs=custom_model_kwargs,
    path_to_save_model_outputs=path_to_save_model_outputs,
    random_seed=random_seed,
)
df_results = pd.DataFrame(payloads)

In [ ]:
df_results.groupby(["model_name", "political_pole"])["model_response"].mean().reset_index()